# enwiktionary HTML dump

Provided as part of the [Wikimedia Enterprise HTML Dumps](https://dumps.wikimedia.org/other/enterprise_html/). Specifically here: https://dumps.wikimedia.org/other/enterprise_html/runs/. There will be files like the following:

```
enwiktionary-NS0-20250201-ENTERPRISE-HTML.json...> 01-Feb-2025 11:43         15443888277
enwiktionary-NS0-20250201-ENTERPRISE-STATS.json    01-Feb-2025 11:43                 121
enwiktionary-NS10-20250201-ENTERPRISE-HTML.json..> 02-Feb-2025 00:09           168297736
enwiktionary-NS10-20250201-ENTERPRISE-STATS.json   02-Feb-2025 00:09                 121
enwiktionary-NS14-20250201-ENTERPRISE-HTML.json..> 01-Feb-2025 19:05           576717474
enwiktionary-NS14-20250201-ENTERPRISE-STATS.json   01-Feb-2025 19:05                 121
enwiktionary-NS6-20250201-ENTERPRISE-HTML.json...> 02-Feb-2025 04:53               14286
enwiktionary-NS6-20250201-ENTERPRISE-STATS.json    02-Feb-2025 04:53                 121
```

The `NS` in the filenames stands for the MediaWiki namespace:

- NS0: Main namespace (actual dictionary entries)
- NS6: File namespace (used for media like images)
- NS10: Template namespace (for reusable page components)
- NS14: Category namespace (for organizing pages into categories)

We want the one like `enwiktionary-NS0-20250201-ENTERPRISE-HTML.json` which contains the actual dictionary entries.

The file is quite large, 14.4 GB on 2025-02-01. The download is slow and sometimes fails partway through.

Each dump output file consists of a tar.gz archive which, when uncompressed and untarred, contains one file, with a single line per article, in json format. Among the attributes defined in the file are the following below, however to see all attributes please visit the [data dictionary for Wikimedia Enterprise APIs](https://enterprise.wikimedia.com/docs/data-dictionary/):

- `name`: the title of the article
- `identifier`: the page id
- `date modified`: the last time the page was modified
- `version`: a compound structure representing the page revision
- `url`: the full url to the page on the wiki
- `namespace`: a compound structure representing the namespace of the page
- `in_language`: a compound structure representing the language of the wiki that has the page
- `article_body`: has two attributes, `html` and `wikitext`, which contain the html and wikitext of the page, respectively
- `license`: license information for the page

Check back on beta structured contents https://enterprise.wikimedia.com/docs/on-demand/#article-structured-contents-beta -- this is offered through API currently but not available in the dumps yet.


In [ ]:
!mkdir -p enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted
!pigz -dc enwiktionary-NS0-20250201-ENTERPRISE-HTML.json.tar.gz | tar xvf - -C enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted

In [ ]:
from pathlib import Path
import json

n66 = Path(
    "enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted/enwiktionary_namespace_0_66.ndjson"
)

# read lines from the file
with n66.open() as f:
    for line in f.readlines():
        content = json.loads(line)
        if content.get("name") == "test":
            article_body = content.pop("article_body")
            with open("test.html", "w") as f:
                f.write(article_body["html"])
            with open("test.wikitext", "w") as f:
                f.write(article_body["wikitext"])
            with open("test.json", "w") as f:
                f.write(json.dumps(content, indent=2, ensure_ascii=False))

In [ ]:
from lxml import etree
import os


def write_chunk(output_dir, chunk, chunk_index, current_size):
    with open(os.path.join(output_dir, f"chunk_{chunk_index}.xml"), "w") as f:
        f.write("<root>\n")
        f.writelines(chunk)
        f.write("</root>\n")
    print(f"Created chunk_{chunk_index}.xml with size {current_size} bytes")


def split_xml(file_path, output_dir, tag, chunk_size, namespace):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    ns_tag = f"{{{namespace}}}{tag}"
    context = etree.iterparse(file_path, events=("end",), tag=ns_tag)
    chunk = []
    chunk_index = 0
    current_size = 0

    for event, elem in context:
        chunk.append(etree.tostring(elem, pretty_print=True, encoding="unicode"))
        current_size += len(chunk[-1])

        if current_size >= chunk_size:
            write_chunk(output_dir, chunk, chunk_index, current_size)
            chunk = []
            current_size = 0
            chunk_index += 1

        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]

    if chunk:
        write_chunk(output_dir, chunk, chunk_index, current_size)


# Example usage
split_xml(
    "test-pages-articles.xml", "output_chunks", "page", 1 * 1024 * 1024, namespace
)  # 1 MB chunks

In [2]:
import sqlite3
import json
import os
import pandas as pd
from glob import glob
import re

# Configuration
# DATA_DIR = "test_ndjson"
DATA_DIR = "enwiktionary-NS0-20250201-ENTERPRISE-HTML"
DB_FILE = "pages.db"
DUPLICATES_CSV = "duplicate_names.csv"


# Connect to SQLite
def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA temp_store=MEMORY;")
    return conn


# Create table
def create_table(conn):
    with conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS pages (
                name TEXT,
                identifier INTEGER,
                date_created TEXT,
                date_modified TEXT,
                date_previously_modified TEXT,
                version TEXT, -- JSON object
                previous_version TEXT, -- JSON object
                url TEXT,
                namespace TEXT, -- JSON object
                in_language TEXT, -- JSON object
                additional_entities TEXT, -- JSON array
                categories TEXT, -- JSON array
                templates TEXT, -- JSON array
                is_part_of TEXT, -- JSON object
                article_body TEXT, -- JSON object
                license TEXT, -- JSON array
                event TEXT, -- JSON object
                image TEXT -- JSON object
            );
        """
        )
        conn.execute("CREATE INDEX IF NOT EXISTS idx_pages_name ON pages (name);")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_pages_url ON pages (url);")


# Insert records in batches
def insert_records(conn, records):
    with conn:
        conn.executemany(
            """
            INSERT INTO pages (
                name, identifier, date_created, date_modified, date_previously_modified,
                version, previous_version, url, namespace, in_language, additional_entities,
                categories, templates, is_part_of, article_body, license, event, image
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """,
            records,
        )


# Write duplicates report before deletion using Pandas
def write_duplicate_report(conn):
    query = """
    SELECT name, COUNT(*) as count
    FROM pages
    GROUP BY name
    HAVING COUNT(*) > 1
    ORDER BY count DESC;
    """

    df = pd.read_sql_query(query, conn)
    df.to_csv(DUPLICATES_CSV, index=False, encoding="utf-8")

    print(f"Duplicate report written to {DUPLICATES_CSV}")


# Remove duplicate names, keeping only the most recent date_modified
def remove_duplicates(conn):
    print("Removing duplicate records, keeping only the latest versions...")
    with conn:
        conn.execute(
            """
            DELETE FROM pages
            WHERE rowid NOT IN (
                SELECT rowid FROM (
                    SELECT rowid, name, MAX(date_modified) AS latest_date
                    FROM pages
                    GROUP BY name
                )
            );
        """
        )
    print("Duplicate records removed.")


# Load all NDJSON files and process them
def load_all_records(delete_after=False):
    """Process all NDJSON files and optionally delete them after successful processing."""
    conn = get_db_connection()
    create_table(conn)

    def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

    ndjson_files = sorted(glob(os.path.join(DATA_DIR, "*.ndjson")), key=natural_sort_key)
    print(f"Found {len(ndjson_files)} NDJSON files.")

    batch = []
    for filepath in ndjson_files:
        print(f"Processing {filepath}...")
        try:
            with open(filepath, "r", encoding="utf-8") as file:
                for line in file:
                    try:
                        data = json.loads(line)
                        record = (
                            data.get("name"),
                            data.get("identifier"),
                            data.get("date_created"),
                            data.get("date_modified"),
                            data.get("date_previously_modified"),
                            json.dumps(data.get("version"), ensure_ascii=False),
                            json.dumps(
                                data.get("previous_version"), ensure_ascii=False
                            ),
                            data.get("url"),
                            json.dumps(data.get("namespace"), ensure_ascii=False),
                            json.dumps(data.get("in_language"), ensure_ascii=False),
                            json.dumps(
                                data.get("additional_entities"), ensure_ascii=False
                            ),
                            json.dumps(data.get("categories"), ensure_ascii=False),
                            json.dumps(data.get("templates"), ensure_ascii=False),
                            json.dumps(data.get("is_part_of"), ensure_ascii=False),
                            json.dumps(data.get("article_body"), ensure_ascii=False),
                            json.dumps(data.get("license"), ensure_ascii=False),
                            json.dumps(data.get("event"), ensure_ascii=False),
                            json.dumps(data.get("image"), ensure_ascii=False),
                        )
                        batch.append(record)

                        if len(batch) >= 500:  # Insert in batches
                            insert_records(conn, batch)
                            batch = []

                    except json.JSONDecodeError as e:
                        print(f"⚠️ Skipping invalid JSON in {filepath}: {e}")

            # Insert any remaining records for this file
            if batch:
                insert_records(conn, batch)
                batch = []

            # Delete the file if requested and processing was successful
            if delete_after:
                try:
                    os.remove(filepath)
                    print(f"🗑️ Deleted {filepath}")
                except Exception as e:
                    print(f"⚠️ Could not delete {filepath}: {e}")

        except Exception as e:
            print(f"❌ Error processing {filepath}: {e}")
            continue

    print("✅ All records inserted.")

    # Write duplicate report before deduplication
    write_duplicate_report(conn)

    # Remove duplicates
    remove_duplicates(conn)

    conn.close()
    print("✅ Database cleaned up.")

    # Try to remove the directory if it's empty and we were deleting files
    if delete_after:
        try:
            os.rmdir(DATA_DIR)
            print(f"🗑️ Removed empty directory {DATA_DIR}")
        except OSError:
            print(f"⚠️ Could not remove {DATA_DIR} (might not be empty)")


# Run the process
load_all_records(delete_after=True)  # Set to True to delete files after processing

Found 67 NDJSON files.
Processing enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_0.ndjson...
🗑️ Deleted enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_0.ndjson
Processing enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_1.ndjson...
🗑️ Deleted enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_1.ndjson
Processing enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_10.ndjson...
🗑️ Deleted enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_10.ndjson
Processing enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_11.ndjson...
🗑️ Deleted enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_11.ndjson
Processing enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_12.ndjson...
🗑️ Deleted enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_12.ndjson
Processing enwiktionary-NS0-20250201-ENTERPRISE-HTML/enwiktionary_namespace_0_13.ndjso

In [13]:
import sqlite3
import json

DB_FILE = "pages.db"


def get_article_html(page_name):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    query = """
    SELECT article_body FROM pages WHERE name = ?;
    """
    cursor.execute(query, (page_name,))
    result = cursor.fetchone()

    conn.close()

    if result:
        article_body = json.loads(result[0])  # Deserialize JSON
        return article_body.get("html")  # Extract the "html" field
    else:
        return None  # Page not found


# Example usage
# page_name = "lime"
# page_name = "㔩"
page_name = "盍"
html_content = get_article_html(page_name)

if html_content:
    print(html_content)
else:
    print(f"No article_body found for page: {page_name}")

<!DOCTYPE html>
<html prefix="dc: http://purl.org/dc/terms/ mw: http://mediawiki.org/rdf/" about="https://en.wiktionary.org/wiki/Special:Redirect/revision/82453774"><head prefix="mwr: https://en.wiktionary.org/wiki/Special:Redirect/"><meta charset="utf-8"/><meta property="mw:pageId" content="21535"/><meta property="mw:pageNamespace" content="0"/><link rel="dc:replaces" resource="mwr:revision/81776018"/><meta property="mw:revisionSHA1" content="fc0c9abfaedc7f76b2a1fe0485bd3ef7ca6d591a"/><meta property="dc:modified" content="2024-10-21T16:03:48.000Z"/><meta property="mw:htmlVersion" content="2.8.0"/><meta property="mw:html:version" content="2.8.0"/><link rel="dc:isVersionOf" href="//en.wiktionary.org/wiki/%E7%9B%8D"/><base href="//en.wiktionary.org/wiki/"/><title>盍</title><meta property="mw:generalModules" content="ext.cite.ux-enhancements"/><meta property="mw:moduleStyles" content="ext.cite.parsoid.styles|ext.cite.styles"/><link rel="stylesheet" href="/w/load.php?lang=en&amp;modules=ext

In [ ]:
from selectolax.parser import HTMLParser

tree = HTMLParser(html_content)

# sections = tree.css('section:has([id^="Etymology"])')
# for section in sections:
#     print(section.text(deep=True))

# # get all headword class elements' text, as well as the part of speech
# headwords = tree.css(".headword")
# for headword in headwords:
#     print(headword.text(deep=True))

# traverse the tree, finding each lang from the text of h2 elements.
# then find each part of speech from the text of the h3 elements which are inside a section element
# that is a sibling of the h2 element
# for n in tree.root.traverse():
#     if n.tag == "h2" and (lang := n.text()):  # in ALLOWED_LANGS:
#         print(lang)
#     if n.tag == "h3" and (ety := n.text()).startswith("Etymology"):
#         print(ety)
#     if n.tag in ("h3", "h4") and (pos := n.text()) in ALLOWED_POS_HEADERS:
#         print(pos)
#     if "headword" in (classes := n.attributes.get("class", "").split()):
#         print(n.text())

lang, ety, pos, term = None, None, None, None

for h2 in tree.css("h2"):
    if not (lang := h2.text()):  # and in ALLOWED_LANGS:
        continue
    print(lang)
    for section in h2.parent.css("section > section"):
        # print(section.id)
        for h3 in section.css("h3"):
            ety_text = h3.text()
            ety_num = ety_text.removeprefix("Etymology")
            if ety_num != ety_text:
                ety = ety_num or "1"
                print(ety)
                if ety_num:
                    for section in h3.parent.css("section > section"):
                        # print(section.id)
                        for h4 in section.css("h4"):
                            if (pos := h4.text()) in ALLOWED_POS_HEADERS:
                                print(pos)
                                term = h4.parent.css_first(".headword").text()
                                print(term)
                                print(lang, ety, pos, term)
            if (pos := h3.text()) in ALLOWED_POS_HEADERS:
                print(pos)
                term = h3.parent.css_first(".headword").text()
                print(term)
                print(lang, ety, pos, term)
            break

# todo:
# - handle when there are no pos headers like with Chinese which goes directly to Definitions header
# - handle when there is no headword like with 盍 Japanese Kanji (default to title)
# - use Glyph Origin when there is no Etymology for CJKV?

Translingual
Han character
盍
Translingual None Han character 盍
Chinese
1
Japanese
Kanji


AttributeError: 'NoneType' object has no attribute 'text'

In [5]:
# https://en.wiktionary.org/wiki/Wiktionary:Entry_layout

ALLOWED_POS_HEADERS = (
    # Parts of speech
    "Adjective",
    "Adverb",
    "Ambiposition",
    "Article",
    "Circumposition",
    "Classifier",
    "Conjunction",
    "Contraction",
    "Counter",
    "Determiner",
    "Ideophone",
    "Interjection",
    "Noun",
    "Numeral",
    "Participle",
    "Particle",
    "Postposition",
    "Preposition",
    "Pronoun",
    "Proper noun",
    "Verb",
    # Morphemes
    "Circumfix",
    "Combining form",
    "Infix",
    "Interfix",
    "Prefix",
    "Root",
    "Suffix",
    # Symbols and characters
    "Diacritical mark",
    "Letter",
    "Ligature",
    "Number",
    "Punctuation mark",
    "Syllable",
    "Symbol",
    # Phrases
    "Phrase",
    "Proverb",
    "Prepositional phrase",
    # Han characters and language-specific varieties
    "Han character",
    "Hanzi",
    "Kanji",
    "Hanja",
    # Other linguistic categories
    "Romanization",
    "Logogram",
    "Determinative",
)

# refs

- https://github.com/tylertitsworth/multi-mediawiki-rag/blob/main/embed.py
- https://huggingface.co/learn/cookbook/en/structured_generation

# preprocesing

- preprocess html to get list of all unique (lang, term, href, title) tuples


In [ ]:
import pandas as pd
import concurrent.futures
from bs4 import BeautifulSoup
import os
import json
from pathlib import Path


def find_term_links(html_content):
    soup = BeautifulSoup(html_content, "lxml")
    term_links = set()

    for i_element in soup.find_all("i", lang=True):
        lang = i_element.get("lang", "")
        for a_element in i_element.find_all("a"):
            term = a_element.get_text(strip=True)
            href = a_element.get("href", "")
            title = a_element.get("title", "")
            term_links.add((lang, term, href, title))

    return term_links


def process_ndjson_file(ndjson_file_path):
    term_links = set()

    with open(ndjson_file_path, "r") as f:
        for line in f:
            content = json.loads(line)
            article_body = content.get("article_body", {})
            html_content = article_body.get("html", "")
            if html_content:
                term_links.update(find_term_links(html_content))

    return term_links


def process_all_files(file_paths):
    all_term_links = set()

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = (
            executor.submit(process_ndjson_file, file_path) for file_path in file_paths
        )
        for future in concurrent.futures.as_completed(futures):
            term_links = future.result()
            all_term_links.update(term_links)

    return all_term_links


# Example usage
file_paths = Path("enwiktionary-NS0-20250201-ENTERPRISE-HTML-extracted").glob(
    "*.ndjson"
)
all_term_links = process_all_files(file_paths)

# Write to CSV, sort by lang and term
df = pd.DataFrame(all_term_links, columns=["lang", "term", "href", "title"])
df.sort_values(["lang", "term"], inplace=True)
df.to_csv("term_links.csv", index=False)

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("term_links.csv")

# Group by 'lang' and 'term' and filter groups with more than one unique 'href'
duplicate_groups = df.groupby(["lang", "term"]).filter(
    lambda x: x["href"].nunique() > 1
)

# # Display the duplicate groups
# print(duplicate_groups)

# Write the duplicate groups to a CSV file
duplicate_groups.to_csv("term_links_duplicates.csv", index=False)

        lang   term                                  href  title
20        aa    -ta                           ./-ytu#Afar   -ytu
21        aa    -ta                            ./-ta#Afar    -ta
23        aa    -tu                           ./-ytu#Afar   -ytu
24        aa    -tu                            ./-tu#Afar    -tu
25        aa    -tá                            ./-ta#Afar    -ta
...      ...    ...                                   ...    ...
3102335   zu   umu-                           ./umu-#Zulu   umu-
3102336   zu   umu-                  ./umu-#Zulu:_class_3   umu-
3102532  zza  dayen  ./dayen?action=edit&redlink=1#Zazaki  dayen
3102533  zza  dayen                         ./dayi#Zazaki   dayi
3102534  zza  dayen                         ./daye#Zazaki   daye

[68030 rows x 4 columns]


In [6]:
duplicate_groups.to_csv("term_links_duplicates.csv", index=False)

In [ ]:
from bs4 import BeautifulSoup
import re


def render_html_to_text(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    text = soup.get_text()
    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


text_content = render_html_to_text(html_content)
print(text_content)

In [2]:
from typing import List, Optional
from pydantic import BaseModel


class Item(BaseModel):
    etyNum: int
    lang: str
    term: str
    imputed: bool
    reconstructed: bool
    pos: Optional[List[str]] = None
    gloss: Optional[List[str]] = None
    romanization: Optional[str] = None


class Etymology(BaseModel):
    item: Item
    etyMode: Optional[str] = None
    etyOrder: int
    parents: List["Etymology"] = []


ety_schema = Etymology.model_json_schema()

In [4]:
ety_schema

{'$defs': {'Etymology': {'properties': {'item': {'$ref': '#/$defs/Item'},
    'etyMode': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Etymode'},
    'etyOrder': {'title': 'Etyorder', 'type': 'integer'},
    'parents': {'default': [],
     'items': {'$ref': '#/$defs/Etymology'},
     'title': 'Parents',
     'type': 'array'}},
   'required': ['item', 'etyOrder'],
   'title': 'Etymology',
   'type': 'object'},
  'Item': {'properties': {'etyNum': {'title': 'Etynum', 'type': 'integer'},
    'lang': {'title': 'Lang', 'type': 'string'},
    'term': {'title': 'Term', 'type': 'string'},
    'imputed': {'title': 'Imputed', 'type': 'boolean'},
    'reconstructed': {'title': 'Reconstructed', 'type': 'boolean'},
    'pos': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'},
      {'type': 'null'}],
     'default': None,
     'title': 'Pos'},
    'gloss': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'},
      {'type': 'null'}],
     'defaul

In [3]:
prompt_template = """
You are an expert in word etymologies. Given a word's etymology, your task is to convert it to JSON format. Here is the word and its etymology:

{term} ({lang} {pos})

{ety}
"""

In [4]:
prompt = prompt_template.format(
    term="hap",
    lang="English",
    pos="noun",
    ety="From Middle English hap, happe (“chance, hap, luck, fortune”), potentially cognate with or from Old English ġehæp (“fit, convenient”) and/or Old Norse happ (“hap, chance, good luck”), from Proto-Germanic *hampą (“convenience, happiness”), from Proto-Indo-European *kob- (“good fortune, prophecy; to bend, bow, fit in, work, succeed”).",
)

In [1]:
# model_name = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
model_name = "Orion-zhen/Qwen3-8B-AWQ"
max_model_len = 4096

print(
    f"in separate terminal, run:\n\nvllm serve {model_name} --max-model-len {max_model_len}"
)

in separate terminal, run:

vllm serve Orion-zhen/Qwen3-8B-AWQ --max-model-len 4096


In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="-",
)

completion = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": prompt,
            # "content": "hi",
        }
    ],
    extra_body={"guided_json": ety_schema},
)
print(completion.choices[0].message.content)

{


  "item": {
    "etyNum": 1,
    "lang": "en",
    "term": "hap",
    "imputed": false,
    "reconstructed": false,
    "pos": [
      "noun"
    ],
    "gloss": [
      "chance, luck, fortune"
    ],
    "romanization": "hap"
  },
  "etyMode": "etymology",
  "etyOrder": 1,
  "parents": [
    {
      "item": {
        "etyNum": 2,
        "lang": "en",
        "term": "happe",
        "imputed": false,
        "reconstructed": false,
        "pos": [
          "noun"
        ],
        "gloss": [
          "chance, hap, luck, fortune"
        ],
        "romanization": "happe"
      },
      "etyOrder": 2,
      "parents": [
        {
          "item": {
            "etyNum": 3,
            "lang": "en",
            "term": "Old English ġehæp",
            "imputed": false,
            "reconstructed": false,
            "pos": [
              "noun"
            ],
            "gloss": [
              "fit, convenient"
            ],
            "romanization": "ġehæp"
          },